### 🐍 Python Prerequisite: Type Annotations vs. Runtime Behavior

Standard Python includes **type hints**, but Python's runtime **never enforces them**:

```python
def chat(message: str, temperature: float):
    return f"Message: {message}, Temp: {temperature}"

# Python runs this without complaining, even though types are mismatched!
chat(message=123, temperature="high")
```

Pydantic bridges this gap by reading type annotations dynamically at runtime using `pydantic_core` (written in Rust) to enforce types when objects are created.

## 1. Why Pydantic Exists: Parsing & Validation in One Step

When raw external data arrives (e.g., from an HTTP request, JSON file, or LLM payload), it must be both parsed into Python native objects and validated against domain rules.

Pydantic collapses parsing and validation into **one model class, executed automatically upon instantiation**.

> **Resource:** Read more on [Pydantic BaseModel Documentation](https://docs.pydantic.dev/latest/concepts/models/).

In [6]:
from pydantic import BaseModel

class Product(BaseModel):
    name: str
    price: float
    tags: list[str]
    metadata: dict[str, str]


iphone = Product(name="Iphone", price="12.34", tags=["iphone", 'mobile phone'], metadata={"country": "IN"})

print(iphone.price)


12.34


In [9]:
from pydantic import BaseModel, ValidationError

class ChatRequest(BaseModel):
    message: str
    temperature: float
    model: str
    # dictionary: dict[str, int]

# Automatic type coercion when compatible (e.g., string "0.7" converted to float 0.7)
valid_req = ChatRequest(message="Explain vector databases", temperature="0.7", model="gpt-4o")
print(f"Valid request object created: {valid_req}")
print(f"Temperature type is float: {type(valid_req.temperature)}")

# Automatic validation error on incompatible data types
try:
    ChatRequest(message=123, temperature="invalid_number", model="gpt-4o")
except ValidationError as e:
    print("\n--- Caught expected Pydantic ValidationError ---")
    print(e)

Valid request object created: message='Explain vector databases' temperature=0.7 model='gpt-4o'
Temperature type is float: <class 'float'>

--- Caught expected Pydantic ValidationError ---
2 validation errors for ChatRequest
message
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
temperature
  Input should be a valid number, unable to parse string as a number [type=float_parsing, input_value='invalid_number', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/float_parsing


## 3. Required vs. Optional Fields

Pydantic draws a clear boundary between **Allowed Values** and **Requirement Status**:

1. `float | None` defines **allowed types** (it allows numbers or `None`).
2. `= None` (or any default assignment) specifies **if the caller can omit the key**.

> 🔑 **Rule:** Defining a field as `float | None` *without* a default value means it is **still required**—the caller must explicitly provide `None`.

In [10]:
class ChatRequestOptional(BaseModel):
    message: str                       # Required field
    temperature: float | None = None   # Optional field, defaults to None
    max_tokens: int = 1000             # Optional field, defaults to 1000
    system_prompt: str | None          # Explicitly required field that accepts None!

# Passing required fields
req1 = ChatRequestOptional(message="Hello", system_prompt=None)
print(f"Req 1: {req1}")

# Failing because 'system_prompt' was omitted even though it accepts None
try:
    ChatRequestOptional(message="Hello")
except ValidationError as e:
    print("\n--- Caught error when missing required field 'system_prompt' ---")
    print(e)


Req 1: message='Hello' temperature=None max_tokens=1000 system_prompt=None

--- Caught error when missing required field 'system_prompt' ---
1 validation error for ChatRequestOptional
system_prompt
  Field required [type=missing, input_value={'message': 'Hello'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [13]:
class ModelConfig(BaseModel):
    model: str
    temperature: float

class Message(BaseModel):
    role: str
    content: str

class NestedChatRequest(BaseModel):
    config: ModelConfig
    messages: list[Message]

raw_payload = {
    "config": {"model": "gpt-4o", "temperature": 0.7},
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is RAG?"}
    ]
}

req = NestedChatRequest(**raw_payload)
print(req.config)

model='gpt-4o' temperature=0.7


## 5. Constraint Validation & Aliases with `Field()`

The `Field()` function defines constraints, default values, descriptions, and custom input/output field names (aliases).

| Constraint Parameter | Description | Example |
| :--- | :--- | :--- |
| `min_length` / `max_length` | Restricts string length | `Field(min_length=1, max_length=100)`[cite: 1] |
| `gt` / `ge` | Greater than / Greater than or equal to | `Field(ge=0.0)`[cite: 1] |
| `lt` / `le` | Less than / Less than or equal to | `Field(le=2.0)`[cite: 1] |
| `alias` | Maps external JSON keys to Pythonic field names | `Field(alias="api_key")`[cite: 1] |

In [14]:
from pydantic import Field

class ConstrainedChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=5000)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    # Alias maps incoming key 'user_id' to internal variable name 'user_identifier'
    user_identifier: str = Field(alias="user_id")

# Passing payload using the alias name 'user_id'
req = ConstrainedChatRequest(message="Hello", user_id="usr_12345", temperature=1.4)
print(f"Parsed with Alias -> Internal Attribute: {req}")


Parsed with Alias -> Internal Attribute: message='Hello' temperature=1.4 user_identifier='usr_12345'


In [15]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

class ConstrainedChatRequest(BaseModel):
    message: Annotated[str, Field(min_length=1, max_length=5000)]
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

In [16]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

# ------------------------------------------------------------------
# Reusable Custom Domain Types with Annotated
# ------------------------------------------------------------------
PromptText = Annotated[
    str, 
    Field(min_length=1, max_length=5000, description="The primary user prompt")
]
Temperature = Annotated[
    float, 
    Field(ge=0.0, le=2.0, description="Sampling temperature")
]

# ------------------------------------------------------------------
# Model Definition using Modern Annotated Syntax
# ------------------------------------------------------------------
class ConstrainedChatRequest(BaseModel):
    message: PromptText
    temperature: Temperature = 0.7
    
    # Inline Annotated field with alias mapping
    user_identifier: Annotated[str, Field(alias="user_id")]

req = ConstrainedChatRequest(message="Explain Vector DBs", user_id="usr_12345")
print(f"Parsed with Alias -> Internal Attribute: {req.user_identifier}")
print(f"Default Temperature Applied: {req.temperature}")

Parsed with Alias -> Internal Attribute: usr_12345
Default Temperature Applied: 0.7


In [17]:
from pydantic import field_validator, model_validator

class SignupRequest(BaseModel):
    username: str
    password: str
    confirm_password: str

    # Single Field Transformation / Validation
    @field_validator("username")
    @classmethod
    def validate_username(cls, value: str) -> str:
        if " " in value:
            raise ValueError("Username cannot contain spaces")
        return value.lower()  # Normalizes username to lowercase

    # Cross-Field Validation
    @model_validator(mode="after")
    def verify_passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self


signup = SignupRequest(username="ANUJ_KUMar", password="1234", confirm_password="1234")

print(signup)


username='anuj_kumar' password='1234' confirm_password='1234'


## 7. Specialized Types: `EmailStr`, `SecretStr`, and `ConfigDict`

Pydantic provides rich specialized types out of the box:

* `EmailStr`: Validates email syntax automatically. *(Note: requires `pip install pydantic[email]`)*
* `SecretStr`: Hides API keys or passwords when printing models to avoid logging sensitive data.
* `ConfigDict(extra="forbid")`: Configuration dictionary to reject unexpected input fields.

> **Resource:** Guide on [Pydantic Settings & Special Types](https://docs.pydantic.dev/latest/concepts/types/).

In [19]:
from pydantic import ConfigDict, SecretStr, EmailStr

class APIConfiguration(BaseModel):
    # Strict configuration: throw error if extra unexpected keys are sent
    model_config = ConfigDict(extra="forbid")
    
    admin_email: EmailStr
    api_key: SecretStr

config = APIConfiguration(
    admin_email="admin@ai-service.com", 
    api_key="sk-proj-secret-token-12345"
)

print(config)


admin_email='admin@ai-service.com' api_key=SecretStr('**********')


## 8. Serialization & Deserialization Pipeline

Pydantic standardizes data transformations across Python objects, dictionaries, and JSON strings[cite: 1]:

```
Dict / DB Row ──────> model_validate() ──────> Pydantic Model Instance
Pydantic Model ─────> model_dump() ──────────> Python Dictionary
Pydantic Model ─────> model_dump_json() ─────> JSON String
```

> **Resource:** See [Pydantic Serialization Docs](https://docs.pydantic.dev/latest/concepts/serialization/).

In [20]:
raw_dict = {"name": "Anuj", "age": 28, "email": "anuj@example.com"}

class User(BaseModel):
    name: str
    age: int
    email: EmailStr

# user = User(**raw_dict)
user = User.model_validate(raw_dict)

user_dict = user.model_dump()

user_json = user.model_dump_json()


print(user)
print(type(user_dict))
print(user_json)

name='Anuj' age=28 email='anuj@example.com'
<class 'dict'>
{"name":"Anuj","age":28,"email":"anuj@example.com"}
